In [72]:
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field
from typing import List, Literal
import json
from openai import OpenAI
from tqdm import tqdm

In [73]:
SYSTEM_PROMPT = """
You are an alarm audit assistant.
Your role is to evaluate whether a human-assigned alarm tag is correct based on alarm context and operator comments.
You must follow the provided tag definitions exactly.
Do not invent new tags or redefine policy.
Output must be valid JSON only.
""".strip()
# USER_PROMPT_TEMPLATE = """
# Tag definitions:
# - Nuisance: environmental triggers (wind, rain, animals)
# - Valid: real incident or confirmed issue (dispatch, verified, fault)
# - FALSE: no issue found

# Task:
# Is the assigned tag correct?

# Return JSON:
# {{"tag_correctness": "Valid|Invalid"}}
# """.strip()

In [74]:
def build_batch_prompt(batch_df: pd.DataFrame) -> str:
    records = []

    for idx, row in batch_df.iterrows():
        records.append({
            "row_id": str(idx),
            "alarm_type": row["Alarm type"],
            "mode": row["Mode"],
            "alarm_name": row["Alarm name"],
            "alarm_point": row["Alarm Point number"],
            "comment": row["Full comment"],
            "tag": row["assigned_tag"],
        })

    return f"""
Tag definitions:
- Nuisance: environmental triggers (wind, rain, animals)
- Valid: real incident or confirmed issue (dispatch, verified, fault)
- FALSE: no issue found

Task:
For each record, determine whether the assigned tag is correct according to the definitions.

Decision rule:
- If the assigned tag matches the situation described by the alarm context and comment, return "Valid"
- If the assigned tag does not match the situation described by the alarm context and comment, return "Invalid"

Instructions:
- Evaluate each record independently
- Use the alarm type, mode, alarm name, and comment together
- Do not default all records to "Invalid"
- Do not default all records to "Valid"
- Return one result for every input record
- Output JSON only

Return JSON only as a list in exactly this format:
[
  {{"row_id": "0", "tag_correctness": "Valid"}},
  {{"row_id": "1", "tag_correctness": "Invalid"}}
]

Records:
{json.dumps(records, ensure_ascii=False)}
""".strip()

In [75]:
import numpy as np
import pandas as pd

def normalize_tag(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    if x.upper() == "FALSE":
        return "FALSE"
    if x.lower() == "valid":
        return "Valid"
    if x.lower() == "nuisance":
        return "Nuisance"
    return x

def prepare_eval_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Ensure clean index for batching
    df = df.reset_index(drop=True)

    # If visible Tag exists but assigned_tag does not, use it
    if "assigned_tag" not in df.columns and "Tag" in df.columns:
        df["assigned_tag"] = df["Tag"]

    df["assigned_tag"] = df["assigned_tag"].map(normalize_tag)
    df["should_be_tag"] = df["should_be_tag"].map(normalize_tag)

    # Handle missing values
    df["assigned_tag"] = df["assigned_tag"].fillna("UNKNOWN")
    df["should_be_tag"] = df["should_be_tag"].fillna("UNKNOWN")
    df["Full comment"] = df["Full comment"].fillna("")

    # Derive mislabeled if not already present
    if "is_mislabeled" not in df.columns:
        df["is_mislabeled"] = df["assigned_tag"] != df["should_be_tag"]

    # Gold label (same as old setup)
    df["Correct Tag"] = np.where(df["is_mislabeled"], "Invalid", "Valid")

    return df

In [76]:
class Evidence(BaseModel):
    field: str
    quote: str
    why: str

class AuditResult(BaseModel):
    tag_correctness: Literal["Valid", "Invalid"]
    confidence: float = Field(ge=0.0, le=1.0)
    evidence: List[Evidence]

In [77]:
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

def run_llm_eval_batched(
    df: pd.DataFrame,
    model_name: str,
    batch_size: int = 20,
    save_every: int = 10,
    checkpoint_path: str = "batched_eval_checkpoint.csv",
) -> pd.DataFrame:
    results = []

    num_batches = (len(df) + batch_size - 1) // batch_size

    for batch_num, start in enumerate(tqdm(range(0, len(df), batch_size), total=num_batches, desc="Running batched LLM eval"), start=1):
        batch_df = df.iloc[start:start + batch_size]
        prompt = build_batch_prompt(batch_df)

        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=max(200, batch_size * 20),
        )

        raw_text = response.choices[0].message.content

        try:
            parsed = json.loads(raw_text)
            parsed_map = {
                str(item["row_id"]): item.get("tag_correctness", "Invalid")
                for item in parsed
                if isinstance(item, dict) and "row_id" in item
            }
        except Exception:
            parsed_map = {}

        for idx, row in batch_df.iterrows():
            results.append({
                **row.to_dict(),
                "model_tag_correctness": parsed_map.get(str(idx), "Invalid"),
                "raw_model_output": raw_text,
            })

        if batch_num % save_every == 0:
            pd.DataFrame(results).to_csv(checkpoint_path, index=False)

    results_df = pd.DataFrame(results)
    results_df.to_csv(checkpoint_path, index=False)
    return results_df

In [78]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import pandas as pd

def normalize_correctness_label(x):
    if pd.isna(x):
        return "Invalid"
    x = str(x).strip()

    if x.lower() == "valid":
        return "Valid"
    if x.lower() == "invalid":
        return "Invalid"

    # handle common sloppy outputs
    if x.lower().startswith("valid"):
        return "Valid"
    if x.lower().startswith("invalid"):
        return "Invalid"

    return "Invalid"

def score_results(results_df: pd.DataFrame):
    y_true = results_df["Correct Tag"].map(normalize_correctness_label)
    y_pred = results_df["model_tag_correctness"].map(normalize_correctness_label)

    acc = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        pos_label="Invalid",
        average="binary",
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=["Valid", "Invalid"]
    )

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1       : {f1:.4f}")
    print("\nConfusion Matrix [Valid, Invalid]:")
    print(cm)

    return {
        "accuracy": acc,
        "precision_invalid": precision,
        "recall_invalid": recall,
        "f1_invalid": f1,
        "confusion_matrix": cm
    }

In [79]:
df = pd.read_csv("synthetic_alarm_logs_with_hidden_columns.csv")
eval_df = prepare_eval_df(df)

results_df = run_llm_eval_batched(
    eval_df,
    model_name="llama3",
    batch_size=20,
    save_every=10,
    checkpoint_path="llama3_eval_checkpoint.csv"
)

metrics = score_results(results_df)

Running batched LLM eval: 100%|██████████| 126/126 [09:05<00:00,  4.33s/it]

Accuracy : 0.7169
Precision: 0.2103
Recall   : 0.7892
F1       : 0.3321

Confusion Matrix [Valid, Invalid]:
[[1617  661]
 [  47  176]]


In [80]:
results_df["model_tag_correctness"].value_counts(dropna=False)
print(results_df["raw_model_output"].head(5).to_string(index=False))

[\n  {"row_id": "0", "tag_correctness": "Valid"...
[\n  {"row_id": "0", "tag_correctness": "Valid"...
[\n  {"row_id": "0", "tag_correctness": "Valid"...
[\n  {"row_id": "0", "tag_correctness": "Valid"...
[\n  {"row_id": "0", "tag_correctness": "Valid"...
